In [ ]:
#Load 3 daily csv's : the first day, a middle day and the last day.
#inspect at least 3 dated batches.
#Taking days far apart lets us notice if anything changes over time.
from pathlib import Path
import pandas as pd
RAW_DIR=Path("../data/raw")
DATES = ["2026-10-01", "2026-10-18", "2026-11-15"]
batches={}
for date in DATES:
    file=RAW_DIR/ f"orders_{date}.csv"
    batches[date]=pd.read_csv(file)
    print(date,batches[date].shape)


In [ ]:
#Check column types
batches["2026-10-01"].dtypes
#order_date is text,  in Silver has to be converted.

In [ ]:
#Count missing values
sample=pd.concat(batches.values(),ignore_index=True)
print("total rows:", len(sample))
sample.isnull().sum()


In [ ]:
#Find invalid values
#Find invalid values
print(sample.describe().T[["min", "max"]])
print()
print("distance_km < 0:", (sample["distance_km"] < 0).sum())
print("items_count == 0:", (sample["items_count"] == 0).sum())
print("order_value_egp < 0:", (sample["order_value_egp"] < 0).sum())

In [ ]:
#Find duplicate orders
print(" duplicate rows:", sample.duplicated().sum())


In [ ]:
#Check the label balance
print(sample["late_delivery"].value_counts())
print(sample["late_delivery"].value_counts(normalize=True).round(3))

In [ ]:
#Check category spellings
for col in ["customer_region", "warehouse_id", "shipping_method", "weather_risk_level"]:
    print(sample[col].value_counts(dropna=False))
    print()

In [ ]:
#Check the dates
for date, df in batches.items():
    wrong = (df["order_date"] != date).sum()
    print(date, "rows with a different order_date:", wrong)

## Stage A findings

Batches inspected: 2026-10-01, 2026-10-18, 2026-11-15 (683 rows, 15 columns).

### found

1. **Column roles**
    1. Identifier: `order_id`
    2. Date: `order_date` (stored as text; must equal the batch date)
    3. Target: `late_delivery` (0 = on time, 1 = late)
    4. Numeric features:
         order_hour, distance_km,
         items_count,
         package_weight_kg,
         order_value_egp,
         promised_delivery_days,
         warehouse_load_pct,
         carrier_on_time_rate
    5. Categorical features:
     customer_region,
     warehouse_id,
     shipping_method,
     weather_risk_level
2. **Missing values** (all ≤ 2%): 
   distance_km 14,
   warehouse_load_pct 13,
   carrier_on_time_rate 10, 
   weather_risk_level 9,
   shipping_method 3. ID, 
   date and label are never missing.
3. **Invalid values:**
   distance_km < 0 in 6 rows,
   items_count = 0 in 5 rows,
   order_value_egp < 0 in 1 row.
All other columns are within their valid ranges.
4. **Duplicates:** 9 order_ids appear twice; every pair is an exact copy.
5. **Categories:** shipping_method has 3 spelling variants (" standard", "EXPRESS", "same_day").
 The other category columns are clean.
6. **Dates:** every order_date matches its batch date.
7. **Label balance:** 73.6% on time, 26.4% late, so accuracy alone is misleading.

### decided

1. **Silver (fixed rules, nothing learned from data)**
    1. Drop duplicate order_ids, keeping the first row.
    2. Trim spaces and map shipping_method to Standard / Express / Same_Day.
    3. Convert order_date from string to date.
    4. Turn invalid or out-of-range numbers into missing values.
    5. Leave missing values missing.
2. **ML training (learned from training data only)**
    1. Fill missing numbers with the median, missing categories with the most frequent value.
    2. One-hot encode categories and scale numbers.
    3. Use a stratified train/test split.
    4. Evaluate with precision, recall and F1 for "late del.".
3. **Gold validation (fail clearly on)**
    1. Empty data
    2. Missing required columns
    3. Duplicate order_ids
    4. An order_date different from the batch date
    5. A label that isn't 0 or 1